# Model Comparison — MLX (Apple Silicon)

Runs entirely on-device via `mlx-lm`. Evaluates 3 base models + 3 LoRA-adapted variants across 38 symbols (2022–2025).

**One-time setup:** the conversion cells below merge LoRA adapters into their base models and convert everything to quantized MLX format. After that, inference is fast and fully offline.

### Installations

In [24]:
%pip install mlx-lm transformers peft safetensors huggingface_hub -q
%pip install backtrader alpaca_trade_api plotly pandas psutil -q
%pip install "nbformat>=4.2.0" -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Secrets

In [25]:
HF_TOKEN=''
ALPACA_API_KEY=''
ALPACA_SECRET_KEY=''

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Configuration

In [26]:
MODEL_CONFIGS = [
    # {
    #     'label':       'Qwen2.5-7B base',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    # {
    #     'label':       'Qwen2.5-7B LoRA v1-500',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit',
    #     'mlx_adapter': 'mlx_adapters/qwen7b_lora',
    #     'base_label':  'Qwen2.5-7B base',
    # },
    # {
    #     'label':       'Llama-3.1-8B base',
    #     'mlx_id':      'mlx-community/Meta-Llama-3.1-8B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    # {
    #     'label':       'Llama-3.1-8B LoRA v1-500',
    #     'mlx_id':      'mlx-community/Meta-Llama-3.1-8B-Instruct-4bit',
    #     'mlx_adapter': 'mlx_adapters/llama8b_lora',
    #     'base_label':  'Llama-3.1-8B base',
    # },
    # {
    #     'label':       'Qwen2.5-32B base',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-32B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    {
        'label':       'Qwen2.5-32B LoRA v1-500',
        'mlx_id':      'mlx-community/Qwen2.5-Coder-32B-Instruct-4bit',
        'mlx_adapter': 'mlx_adapters/qwen32b_lora',
        'base_label':  'Qwen2.5-32B base',
    },
]

# TEST_START = '2022-01-01'
# TEST_END   = '2025-12-31'
N_SAMPLES  = 5


# SYMBOLS = [
#     'AAPL', 'AMGN', 'AXP',  'BA',   'CAT',
#     'CRM',  'CSCO', 'CVX',  'DIS',  'DOW',
#     'GS',   'HD',   'HON',  'IBM',  'JNJ',
#     'JPM',  'KO',   'MCD',  'MMM',  'MRK',
#     'MSFT', 'NKE',  'NVDA', 'PG',   'TRV',
#     'UNH',  'V',    'VZ',   'WBA',  'WMT',
#     'AMZN', 'COIN', 'GE',   'GOOGL','NFLX',
#     'NIO',  'TSLA', 'UVV',
# ]

SYMBOLS = ['AAPL', 'AMZN', 'MSFT', 'TSLA', 'GOOGL'] # BENCHMARK_SYMBOLS
TEST_START   = '2022-06-01' # BENCHMARK_START
TEST_END     = '2024-01-01' # BENCHMARK_END


### One-Time Model Conversion

Converts each model to 4-bit quantized MLX format and saves locally.
Subsequent runs skip configs whose `mlx_path` already exists.

> **32B LoRA note:** merging a 32B peft adapter requires ~64 GB RAM (float16 weights).
> If you have 32 GB, only the base 32B model can be converted here.
> To get the 32B LoRA: merge on Colab/Kaggle A100, download the merged MLX folder, place it at `mlx_models/qwen32b_lora`.

In [27]:
# One-time setup: run convert_lora_adapters.ipynb to create mlx_adapters/ before evaluating LoRA configs.

In [28]:
import os
for cfg in MODEL_CONFIGS:
    adapter = cfg.get('mlx_adapter')
    if adapter:
        status = 'ready' if os.path.exists(adapter) else 'MISSING — run convert_lora_adapters.ipynb'
        print(f"  {cfg['label']}: adapter {adapter} [{status}]")
    else:
        print(f"  {cfg['label']}: base model ({cfg['mlx_id']})")

  Qwen2.5-32B LoRA v1-500: adapter mlx_adapters/qwen32b_lora [ready]


### Core Classes

In [29]:
import re
import json
import os
import gc
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from datetime import datetime

import backtrader as bt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

from mlx_lm import load as mlx_load, generate as mlx_gen
from alpaca_trade_api.rest import REST, TimeFrame
from IPython.display import Image, display

In [30]:
from mlx_lm.sample_utils import make_sampler

class MLXModel:
    def __init__(self, mlx_path, adapter_path=None, max_tokens=512, temp=1.0):
        self.max_tokens = max_tokens
        self.sampler = make_sampler(temp=temp)
        print(f'Loading {mlx_path} ...')
        self.model, self.tokenizer = mlx_load(mlx_path, adapter_path=adapter_path)
        if adapter_path:
            print(f'Adapter: {adapter_path}')
        print('Ready.')

    def generate(self, inputs):
        if hasattr(self.tokenizer, 'apply_chat_template'):
            prompt = self.tokenizer.apply_chat_template(
                [{'role': 'user', 'content': inputs.strip()}],
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = inputs.strip()
        return mlx_gen(
            self.model, self.tokenizer,
            prompt=prompt,
            max_tokens=self.max_tokens,
            sampler=self.sampler,
            verbose=False,
        )

    def unload(self):
        del self.model, self.tokenizer
        gc.collect()
        print('Model unloaded.')

In [31]:
class Backtrader:
    _instance = None
    _data_cache = {}

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if hasattr(self, '_initialized'):
            return
        self._initialized = True
        self.rest_api = REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, 'https://paper-api.alpaca.markets')

    def _get_bars(self, symbol, timeframe, start, end):
        key = (symbol, str(timeframe), start, end)
        if key not in Backtrader._data_cache:
            Backtrader._data_cache[key] = self.rest_api.get_bars(
                symbol, timeframe, start, end, adjustment='all'
            ).df
        return Backtrader._data_cache[key]

    def load_bars(self, symbols, start, end, timeframe=TimeFrame.Day):
        print(f'Pre-fetching {len(symbols)} symbols ({start} to {end})...')
        for i, sym in enumerate(symbols, 1):
            self._get_bars(sym, timeframe, start, end)
            print(f'  [{i}/{len(symbols)}] {sym}')
        print(f'Done. {len(Backtrader._data_cache)} series cached.')

    def run_backtest(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10000, plot=False):
        cerebro = bt.Cerebro(stdstats=True)
        cerebro.broker.setcash(cash)
        cerebro.addstrategy(strategy)
        cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='mysharpe')
        cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annual_return')
        cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

        if isinstance(symbols, str):
            cerebro.adddata(bt.feeds.PandasData(
                dataname=self._get_bars(symbols, timeframe, start, end), name=symbols
            ))
        else:
            for sym in symbols:
                cerebro.adddata(bt.feeds.PandasData(
                    dataname=self._get_bars(sym, timeframe, start, end), name=sym
                ))

        init_val = cerebro.broker.getvalue()
        results = cerebro.run()
        _return = (cerebro.broker.getvalue() / init_val - 1) * 100

        strat = results[0]
        sharpe = strat.analyzers.mysharpe.get_analysis().get('sharperatio')
        annual = strat.analyzers.annual_return.get_analysis()
        avg_annual = (sum(annual.values()) / len(annual) * 100) if annual else 0.0
        max_dd = strat.analyzers.drawdown.get_analysis()['max']['drawdown']

        if plot and _return > 0:
            cerebro.plot(iplot=False)
            for i, fig_num in enumerate(plt.get_fignums(), start=1):
                plt.figure(fig_num)
                fname = f'backtest_plot_{i}.png'
                plt.savefig(fname, dpi=140, bbox_inches='tight')
                display(Image(fname))
            plt.close('all')

        return _return, sharpe, avg_annual, max_dd

    def timed_backtest(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10000):
        with ThreadPoolExecutor(max_workers=1) as ex:
            future = ex.submit(self.run_backtest, strategy, symbols, start, end, timeframe, cash, False)
            try:
                return future.result(timeout=10)
            except FuturesTimeout:
                raise TimeoutError('Backtest timed out')

In [32]:
try:
    from unsloth import check_python_modules
except ImportError:
    import ast
    def check_python_modules(code):
        try:
            ast.parse(code)
            return True, {}
        except SyntaxError as e:
            return False, {'error': str(e)}

class RewardFunctions:

    def extract_function(text):
        if text.count('```') >= 2:
            first = text.find('```') + 3
            second = text.find('```', first)
            fx = text[first:second].strip().removeprefix('python\n')
            fx = fx[fx.find('class Strategy'):]
            if fx.startswith('class Strategy(bt.Strategy):'):
                return fx
        idx = text.find('class Strategy(bt.Strategy):')
        if idx != -1:
            return text[idx:]
        return None

    def function_works(function):
        if function is None:
            return False
        ok, info = check_python_modules(function)
        return not (ok is False or 'error' in info)

    def has_required_functions(text):
        has_init = bool(re.search(r'def\s+__init__\s*\([^)]*\)\s*:', text))
        has_next = bool(re.search(r'def\s+next\s*\([^)]*\)\s*:', text))
        return has_init and has_next

    def extract_strategy(func):
        namespace = {'bt': bt}
        exec(func, namespace)
        return namespace['Strategy']

### Evaluation Functions

In [33]:
def make_prompt(symbol, start, end):
    lines = [
        f'Create a trading strategy for {symbol} from {start} to {end} that is fully compatible with the following backtesting setup:',
        '',
        '- Framework: Backtrader',
        '- Strategy must subclass bt.Strategy',
        '- The strategy will be passed directly into:',
        'run_backtest(StrategyClass, symbols, start, end, timeframe, cash)',
        '',
        'STRICT RULES:',
        '1. Output ONLY a single Python class definition (no explanations, no markdown, no comments outside the class).',
        '2. The class MUST be named Strategy.',
        '3. Do NOT include imports (bt is already available).',
        '4. Do NOT reference external data, files, APIs, or indicators outside Backtrader.',
        '5. The strategy MUST work for single-symbol backtests.',
        '6. All indicators must be created in __init__.',
        '7. Trading logic must be implemented in next().',
        '8. Orders must use only: self.buy(), self.sell(), self.close(), self.order_target_percent().',
        '9. No plotting, printing, logging, or analyzers.',
        '10. Strategy must be deterministic and backtest-safe (no lookahead bias).',
        '',
        'Return ONLY the Python class. DO NOT output anything else.',
    ]
    return '\n'.join(lines)

In [34]:
def evaluate_symbol(model, bt_instance, symbol, start, end, n_samples):
    prompt = make_prompt(symbol, start, end)
    results = []

    for i in range(n_samples):
        rec = {
            'symbol': symbol, 'sample': i + 1,
            'status': None, 'reward_score': None,
            'return_pct': None, 'sharpe_ratio': None,
            'avg_annual_return_pct': None, 'max_drawdown_pct': None,
            'strategy_code': None,
        }
        try:
            raw = model.generate(prompt)
            func = RewardFunctions.extract_function(raw)
            rec['strategy_code'] = func

            if not RewardFunctions.has_required_functions(func or ''):
                rec.update({'status': 'missing_methods', 'reward_score': -10})
                results.append(rec)
                continue

            if not RewardFunctions.function_works(func):
                rec.update({'status': 'invalid_code', 'reward_score': -3})
                results.append(rec)
                continue

            strategy_cls = RewardFunctions.extract_strategy(func)
            _ret, sharpe, avg_ann, max_dd = bt_instance.timed_backtest(
                strategy_cls, symbol, start, end
            )
            rec['return_pct']           = _ret
            rec['sharpe_ratio']          = sharpe
            rec['avg_annual_return_pct'] = avg_ann
            rec['max_drawdown_pct']      = max_dd

            if _ret == 0 and sharpe is None:
                rec.update({'status': 'no_trades',  'reward_score': -1})
            elif _ret > 0:
                rec.update({'status': 'profitable', 'reward_score': max(avg_ann, 1)})
            else:
                rec.update({'status': 'loss',       'reward_score': 0})

        except TimeoutError:
            rec.update({'status': 'timeout',   'reward_score': -2})
        except Exception as e:
            rec.update({'status': 'exception', 'reward_score': -2})
            print(f'    [{symbol} s{i+1}] {str(e)[:100]}')

        results.append(rec)

    scores   = [r['reward_score'] for r in results if r['reward_score'] is not None]
    statuses = [r['status'] for r in results]
    print(f'  {symbol}: {statuses}  scores={scores}')
    return results

In [35]:
def evaluate_model(config, bt_instance, symbols=None, start=TEST_START, end=TEST_END, n_samples=N_SAMPLES):
    symbols      = symbols or SYMBOLS
    label        = config['label']
    mlx_id       = config['mlx_id']
    adapter_path = config.get('mlx_adapter')

    if adapter_path and not os.path.exists(adapter_path):
        print(f'SKIP {label}: adapter {adapter_path} not found. Run convert_lora_adapters.ipynb first.')
        return []

    print(f"\n{'='*60}")
    print(f'Evaluating: {label}')
    print(f'Model:      {mlx_id}')
    if adapter_path:
        print(f'Adapter:    {adapter_path}')
    print(f'Symbols: {len(symbols)}  |  Samples/symbol: {n_samples}  |  {start} to {end}')
    print(f"{'='*60}")

    model   = MLXModel(mlx_id, adapter_path=adapter_path)
    records = []
    for sym in symbols:
        sym_recs = evaluate_symbol(model, bt_instance, sym, start, end, n_samples)
        for r in sym_recs:
            r['model'] = label
        records.extend(sym_recs)

    model.unload()
    ok = sum(1 for r in records if r['status'] == 'profitable')
    print(f'[{label}] Done. Profitable: {ok}/{len(records)}')
    return records

### Run Evaluation

In [36]:
bt_instance = Backtrader()
bt_instance.load_bars(SYMBOLS, TEST_START, TEST_END)

Pre-fetching 5 symbols (2022-06-01 to 2024-01-01)...
  [1/5] AAPL
  [2/5] AMZN
  [3/5] MSFT
  [4/5] TSLA
  [5/5] GOOGL
Done. 5 series cached.


In [37]:
all_results = []
for config in MODEL_CONFIGS:
    records = evaluate_model(config, bt_instance)
    all_results.extend(records)

df = pd.DataFrame(all_results)
print(f'\nTotal samples collected: {len(df)}')
df.head(10)


Evaluating: Qwen2.5-32B LoRA v1-500
Model:      mlx-community/Qwen2.5-Coder-32B-Instruct-4bit
Adapter:    mlx_adapters/qwen32b_lora
Symbols: 5  |  Samples/symbol: 5  |  2022-06-01 to 2024-01-01
Loading mlx-community/Qwen2.5-Coder-32B-Instruct-4bit ...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 81049.35it/s]


Adapter: mlx_adapters/qwen32b_lora
Ready.
  AAPL: ['no_trades', 'no_trades', 'profitable', 'no_trades', 'profitable']  scores=[-1, -1, 14.059085182575743, -1, 1]
  AMZN: ['profitable', 'profitable', 'profitable', 'profitable', 'no_trades']  scores=[20.7512099103289, 11.606438537843555, 3.668579336266098, 3.6948378471055197, -1]
    [MSFT s4] 'Lines_LineSeries_LineIterator_DataAccessor_Strateg' object has no attribute 'order'
  MSFT: ['profitable', 'profitable', 'profitable', 'exception', 'profitable']  scores=[20.311232098474786, 13.109394423151299, 18.71275, -2, 20.311232098474786]
  TSLA: ['loss', 'loss', 'profitable', 'loss', 'profitable']  scores=[0, 0, 25.413414683987728, 0, 25.350607579869415]
  GOOGL: ['loss', 'loss', 'no_trades', 'profitable', 'no_trades']  scores=[0, 0, -1, 16.111999999999995, -1]
Model unloaded.
[Qwen2.5-32B LoRA v1-500] Done. Profitable: 13/25

Total samples collected: 25


,symbol,sample,status,reward_score,return_pct,sharpe_ratio,avg_annual_return_pct,max_drawdown_pct,strategy_code,model
0,AAPL,1,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
1,AAPL,2,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
2,AAPL,3,profitable,14.059085,27.76220,0.855062,14.059085,19.196279,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
3,AAPL,4,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
4,AAPL,5,profitable,1.000000,0.23690,-1.727900,0.119676,0.484375,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
5,AMZN,1,profitable,20.751210,35.51983,0.615762,20.751210,30.757862,class Strategy(bt.Strategy):\n params = (('...,Qwen2.5-32B LoRA v1-500
6,AMZN,2,profitable,11.606439,8.66484,0.266034,11.606439,32.807918,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
7,AMZN,3,profitable,3.668579,4.26885,0.149111,3.668579,25.690568,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
8,AMZN,4,profitable,3.694838,4.39020,0.152176,3.694838,25.472889,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
9,AMZN,5,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500


In [38]:
df = pd.DataFrame(all_results)
print(f'\nTotal samples collected: {len(df)}')
df.head(10)


Total samples collected: 25


,symbol,sample,status,reward_score,return_pct,sharpe_ratio,avg_annual_return_pct,max_drawdown_pct,strategy_code,model
0,AAPL,1,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
1,AAPL,2,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
2,AAPL,3,profitable,14.059085,27.76220,0.855062,14.059085,19.196279,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
3,AAPL,4,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
4,AAPL,5,profitable,1.000000,0.23690,-1.727900,0.119676,0.484375,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500
5,AMZN,1,profitable,20.751210,35.51983,0.615762,20.751210,30.757862,class Strategy(bt.Strategy):\n params = (('...,Qwen2.5-32B LoRA v1-500
6,AMZN,2,profitable,11.606439,8.66484,0.266034,11.606439,32.807918,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
7,AMZN,3,profitable,3.668579,4.26885,0.149111,3.668579,25.690568,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
8,AMZN,4,profitable,3.694838,4.39020,0.152176,3.694838,25.472889,class Strategy(bt.Strategy):\n def __init__...,Qwen2.5-32B LoRA v1-500
9,AMZN,5,no_trades,-1.000000,0.00000,NaN,0.000000,0.000000,class Strategy(bt.Strategy):\n params = (\n...,Qwen2.5-32B LoRA v1-500


In [39]:
df.to_csv('model_comparison_results_2.csv', index=False)
print('Saved to model_comparison_results_2.csv')

Saved to model_comparison_results_2.csv


### Results Analysis

In [40]:
summary = df.groupby('model').agg(
    total=('sample', 'count'),
    valid=('status', lambda x: x.isin(['profitable', 'loss']).sum()),
    profitable=('status', lambda x: (x == 'profitable').sum()),
    loss=('status', lambda x: (x == 'loss').sum()),
    no_trades=('status', lambda x: (x == 'no_trades').sum()),
    invalid=('status', lambda x: x.isin(['missing_methods', 'invalid_code', 'exception', 'timeout']).sum()),
    
    mean_reward=('reward_score', 'mean'),
    mean_return=('return_pct', 'mean'),
    mean_sharpe=('sharpe_ratio', 'mean'),
    mean_annual=('avg_annual_return_pct', 'mean'),
).reset_index()
summary['profitable_pct'] = (summary['profitable'] / summary['total'] * 100).round(1)
summary['valid_pct'] = (summary['valid'] / summary['total'] * 100).round(1)
display(summary.round(3))

,model,total,valid,profitable,loss,no_trades,invalid,mean_reward,mean_return,mean_sharpe,mean_annual,profitable_pct,valid_pct
0,Qwen2.5-32B LoRA v1-500,25,18,13,5,6,1,7.444,7.801,0.203,10.225,52.0,72.0


In [41]:
status_counts = df.groupby(['model', 'status']).size().reset_index(name='count')
fig = px.bar(
    status_counts, x='model', y='count', color='status', barmode='stack',
    title='Outcome Breakdown by Model',
    color_discrete_map={
        'profitable': '#2ecc71', 'loss': '#e67e22', 'no_trades': '#95a5a6',
        'missing_methods': '#e74c3c', 'invalid_code': '#c0392b',
        'exception': '#9b59b6', 'timeout': '#7f8c8d',
    },
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=30)
fig.show()

In [42]:
best = (
    df[df['reward_score'].notna()]
    .sort_values('reward_score', ascending=False)
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
)

pivot = best.pivot_table(index='symbol', columns='model', values='reward_score')
melted = pivot.reset_index().melt(id_vars='symbol', var_name='model', value_name='best_reward')
fig = px.bar(
    melted, x='symbol', y='best_reward', color='model', barmode='group',
    title='Best Reward Score per Symbol — Model Comparison',
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show()

In [43]:
pivot_ret = best.pivot_table(index='symbol', columns='model', values='return_pct')
for config in MODEL_CONFIGS:
    if not config.get('hf_adapter') or not config.get('base_label'):
        continue
    adapted_label = config['label']
    base_label    = config['base_label']
    if base_label not in pivot_ret.columns or adapted_label not in pivot_ret.columns:
        continue
    delta = (pivot_ret[adapted_label] - pivot_ret[base_label]).to_frame(name='delta_return_pct')
    fig = px.imshow(
        delta.T,
        title=f'Return Delta: {adapted_label} vs {base_label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [44]:
save_dir = 'model_comparison_strategies'
os.makedirs(save_dir, exist_ok=True)
saved = 0

for _, row in best[best['status'] == 'profitable'].iterrows():
    model_dir = os.path.join(save_dir, row['model'].replace('/', '_').replace(' ', '_'))
    os.makedirs(model_dir, exist_ok=True)

    code = row.get('strategy_code')
    if code:
        with open(os.path.join(model_dir, f"{row['symbol']}_strategy.py"), 'w') as f:
            f.write(code)

    stats = {
        'model':                  row['model'],
        'symbol':                 row['symbol'],
        'return_pct':             float(row['return_pct'])             if pd.notna(row['return_pct'])             else None,
        'sharpe_ratio':           float(row['sharpe_ratio'])           if pd.notna(row['sharpe_ratio'])           else None,
        'avg_annual_return_pct':  float(row['avg_annual_return_pct'])  if pd.notna(row['avg_annual_return_pct'])  else None,
        'max_drawdown_pct':       float(row['max_drawdown_pct'])       if pd.notna(row['max_drawdown_pct'])       else None,
    }
    with open(os.path.join(model_dir, f"{row['symbol']}_stats.json"), 'w') as fj:
        json.dump(stats, fj, indent=2)
    saved += 1

print(f'Saved {saved} profitable strategies to {save_dir}/')

Saved 5 profitable strategies to model_comparison_strategies/
